# LightGBM (Multiclass + Ordinal Penalty)
Notebook baseline theo pipeline cua TLSTM, danh gia bang: Accuracy, Macro F1, AUC, QWK.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, RobustScaler, label_binarize
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, cohen_kappa_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import lightgbm as lgb
except Exception:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q'])
    import lightgbm as lgb

def detect_kaggle_runtime() -> bool:
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '').strip():
        return True
    return Path('/kaggle/input').exists() and Path('/kaggle/working').exists()

IN_KAGGLE = detect_kaggle_runtime()

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'src').exists():
            return p
    return start

PROJECT_ROOT = Path('/kaggle/working') if IN_KAGGLE else find_project_root(Path.cwd().resolve())
ARTIFACT_DIR = PROJECT_ROOT / 'credit_rating_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

In [ ]:
FINANCIAL_FEATURES = [
    'current_ratio', 'debt_equity_ratio', 'gross_profit_margin', 'operating_profit_margin',
    'ebit_margin', 'pretax_profit_margin', 'net_profit_margin', 'asset_turnover',
    'roe', 'roa', 'operating_cashflow_ps', 'free_cashflow_ps'
]
TARGET_COL = 'rating_detail'
TARGET_ORDERED_LABELS = ['Distressed', 'HY', 'IG']
INPUT_SIZE = 1
HORIZON = 1

def resolve_split_path(default_path, local_fallbacks=None):
    candidates = [Path(default_path)]
    for p in (local_fallbacks or []):
        p_obj = Path(p)
        candidates.append(PROJECT_ROOT / p_obj if not p_obj.is_absolute() else p_obj)
    if IN_KAGGLE:
        kaggle_root = Path('/kaggle/input')
        expanded = []
        for p in candidates:
            expanded.append(p)
            if not p.exists() and kaggle_root.exists():
                expanded.extend(kaggle_root.rglob(p.name))
        candidates = expanded
    seen = set()
    deduped = []
    for p in candidates:
        p = Path(p)
        key = str(p)
        if key not in seen:
            seen.add(key)
            deduped.append(p)
    for p in deduped:
        if p.exists():
            return p
    raise FileNotFoundError(f'Khong tim thay file split: {deduped}')

TRAIN_PATH = resolve_split_path(
    '/kaggle/input/datasets/tailength/corporate-credit-rating/train_augmented_timegan.csv',
    ['data/processed/train_augmented_timegan.csv']
)
VAL_PATH = resolve_split_path('/kaggle/input/datasets/tailength/corporate-credit-rating/val.csv', ['data/processed/val.csv'])
TEST_PATH = resolve_split_path('/kaggle/input/datasets/tailength/corporate-credit-rating/test.csv', ['data/processed/test.csv'])

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)
train_df['__split__'] = 'train'
val_df['__split__'] = 'val'
test_df['__split__'] = 'test'
df = pd.concat([train_df, val_df, test_df], ignore_index=True)

df = df.dropna(subset=[TARGET_COL]).copy()
target_as_num = pd.to_numeric(df[TARGET_COL], errors='coerce')
if target_as_num.notna().all():
    df[TARGET_COL] = target_as_num.astype(int)
    observed = sorted(df[TARGET_COL].unique().tolist())
    raw_to_id = {int(v): i for i, v in enumerate(observed)}
    id_to_raw = {i: int(v) for v, i in raw_to_id.items()}
    df[TARGET_COL] = df[TARGET_COL].map(raw_to_id).astype(int)
else:
    tgt = df[TARGET_COL].astype(str).str.strip()
    observed = sorted(tgt.unique().tolist())
    ordered = [x for x in TARGET_ORDERED_LABELS if x in observed] if set(observed).issubset(set(TARGET_ORDERED_LABELS)) else observed
    raw_to_id = {v: i for i, v in enumerate(ordered)}
    id_to_raw = {i: v for v, i in raw_to_id.items()}
    df[TARGET_COL] = tgt.map(raw_to_id).astype(int)

n_classes = int(df[TARGET_COL].nunique())
df['rating_date'] = pd.to_datetime(df['rating_date'], errors='coerce', format='mixed')
if 'sector' not in df.columns:
    df['sector'] = 'UNKNOWN'
df['sector'] = df['sector'].fillna('UNKNOWN').astype(str)
sector_encoder = LabelEncoder()
df['sector_id'] = sector_encoder.fit_transform(df['sector'])
n_sectors = int(df['sector_id'].nunique())

train_mask = df['__split__'].eq('train')
stats_ref = df.loc[train_mask].copy() if train_mask.any() else df.copy()
for c in FINANCIAL_FEATURES:
    med = stats_ref[c].median() if stats_ref[c].notna().any() else 0.0
    df[c] = df[c].fillna(float(0.0 if pd.isna(med) else med))
for c in FINANCIAL_FEATURES:
    lo = stats_ref[c].quantile(0.01)
    hi = stats_ref[c].quantile(0.99)
    if pd.notna(lo) and pd.notna(hi):
        df[c] = df[c].clip(float(lo), float(hi))

df = df.sort_values(['ticker', 'rating_date']).reset_index(drop=True)
for c in FINANCIAL_FEATURES:
    df[f'{c}_delta'] = df.groupby('ticker')[c].diff().fillna(0.0)
MODEL_FEATURES = FINANCIAL_FEATURES + [f'{c}_delta' for c in FINANCIAL_FEATURES]

scaler = RobustScaler()
scaler.fit(df.loc[df['__split__'].eq('train'), MODEL_FEATURES].values)
df[MODEL_FEATURES] = scaler.transform(df[MODEL_FEATURES].values)

def build_padded_window(values, target_idx, input_size):
    if target_idx <= 0:
        x_raw = values[:1]
    else:
        x_raw = values[max(0, target_idx - input_size):target_idx]
    if x_raw.shape[0] == 0:
        x_raw = values[:1]
    if x_raw.shape[0] >= input_size:
        return x_raw[-input_size:]
    pad = np.repeat(x_raw[[0]], input_size - x_raw.shape[0], axis=0)
    return np.concatenate([pad, x_raw], axis=0)

def build_sequences(frame, input_size=1, horizon=1):
    out = {'train': [], 'val': [], 'test': []}
    for _, g in frame.groupby('ticker'):
        g = g.sort_values('rating_date').reset_index(drop=True)
        vals = g[MODEL_FEATURES].values.astype(np.float32)
        ys = g[TARGET_COL].values.astype(int)
        sec = g['sector_id'].values.astype(int)
        sp = g['__split__'].astype(str).str.lower().values
        n = len(g)
        if n >= input_size + horizon:
            for i in range(n - input_size - horizon + 1):
                t = i + input_size
                sample = (vals[i:i+input_size], int(ys[i+input_size-1]), int(sec[t]), int(ys[t]))
                out[sp[t]].append(sample)
        else:
            t = n - 1
            x = build_padded_window(vals, t, input_size)
            sample = (x, int(ys[max(0, t-1)]), int(sec[t]), int(ys[t]))
            out[sp[t]].append(sample)
    return out

seqs = build_sequences(df, input_size=INPUT_SIZE, horizon=HORIZON)
for split_name in ['train', 'val', 'test']:
    if len(seqs[split_name]) == 0:
        raise ValueError(f'Split {split_name} khong co mau sau khi tao window; kiem tra lai du lieu va INPUT_SIZE.')

def unpack(samples):
    X = np.stack([s[0] for s in samples]).astype(np.float32)
    last_y = np.array([s[1] for s in samples], dtype=np.int64)
    sector = np.array([s[2] for s in samples], dtype=np.int64)
    y = np.array([s[3] for s in samples], dtype=np.int64)
    return X, last_y, sector, y

X_train, ly_train, sec_train, y_train = unpack(seqs['train'])
X_val, ly_val, sec_val, y_val = unpack(seqs['val'])
X_test, ly_test, sec_test, y_test = unpack(seqs['test'])

def flatten_with_context(X, ly, sec):
    return np.concatenate([X.reshape(X.shape[0], -1), ly[:, None], sec[:, None]], axis=1)

X_train_flat = flatten_with_context(X_train, ly_train, sec_train)
X_val_flat = flatten_with_context(X_val, ly_val, sec_val)
X_test_flat = flatten_with_context(X_test, ly_test, sec_test)

print('Train/Val/Test:', X_train_flat.shape, X_val_flat.shape, X_test_flat.shape)
print('n_classes:', n_classes, '| n_sectors:', n_sectors)

In [ ]:
def softmax_2d(raw):
    z = raw - np.max(raw, axis=1, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=1, keepdims=True)

ORDINAL_LAMBDA = 0.15
class_index = np.arange(n_classes, dtype=np.float64)[None, :]

def to_raw_matrix(preds, n_cls):
    preds = np.asarray(preds)
    if preds.ndim == 2:
        if preds.shape[1] == n_cls:
            return preds
        if preds.shape[0] == n_cls:
            return preds.T
    if preds.ndim == 1:
        if preds.size % n_cls != 0:
            raise ValueError(f'Khong the reshape preds size={preds.size} voi n_cls={n_cls}')
        return preds.reshape(n_cls, -1).T
    raise ValueError(f'Dinh dang preds khong hop le: shape={preds.shape}')

def ordinal_multiclass_objective(preds, train_data):
    y = train_data.get_label().astype(int)
    raw = to_raw_matrix(preds, n_classes)
    p = softmax_2d(raw)

    y_onehot = np.zeros_like(p)
    y_onehot[np.arange(len(y)), y] = 1.0

    exp_rank = np.sum(p * class_index, axis=1, keepdims=True)
    ce_grad = p - y_onehot
    ord_grad = 2.0 * ORDINAL_LAMBDA * (exp_rank - y[:, None]) * p * (class_index - exp_rank)
    grad = ce_grad + ord_grad

    ce_hess = p * (1.0 - p)
    ord_hess = 2.0 * ORDINAL_LAMBDA * np.square(p * (class_index - exp_rank))
    hess = ce_hess + ord_hess + 1e-6

    return grad.T.reshape(-1), hess.T.reshape(-1)

def qwk_eval(preds, train_data):
    y_true = train_data.get_label().astype(int)
    raw = to_raw_matrix(preds, n_classes)
    p = softmax_2d(raw)
    y_pred = np.argmax(p, axis=1)
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    return 'qwk', float(qwk), True

def compute_metrics(y_true, y_pred, proba, n_cls):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average='macro', zero_division=0)
    qwk = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    try:
        y_bin = label_binarize(y_true, classes=list(range(n_cls)))
        auc = roc_auc_score(y_bin, proba, average='macro', multi_class='ovr')
    except Exception:
        auc = float('nan')
    return {'Accuracy': float(acc), 'Macro_F1': float(f1m), 'AUC': float(auc), 'QWK': float(qwk)}

In [ ]:
cls_counts = np.bincount(y_train, minlength=n_classes).astype(float)
cls_weights = np.ones_like(cls_counts, dtype=np.float64)
nz = cls_counts > 0
cls_weights[nz] = cls_counts[nz].sum() / (len(cls_counts[nz]) * cls_counts[nz])
w_train = cls_weights[y_train]

dtrain = lgb.Dataset(X_train_flat, label=y_train, weight=w_train)
dval = lgb.Dataset(X_val_flat, label=y_val, reference=dtrain)

params = {
    'objective': ordinal_multiclass_objective,
    'num_class': n_classes,
    'learning_rate': 0.03,
    'num_leaves': 63,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'min_data_in_leaf': 40,
    'seed': SEED,
    'verbosity': -1,
    'metric': 'None'
}

model = lgb.train(
    params,
    dtrain,
    num_boost_round=1200,
    valid_sets=[dtrain, dval],
    valid_names=['train', 'val'],
    feval=qwk_eval,
    callbacks=[
        lgb.early_stopping(stopping_rounds=120, verbose=True),
        lgb.log_evaluation(period=100),
    ],
)

best_iter = int(model.best_iteration or 1)
curve_step = 1 if best_iter <= 300 else 5
history = {
    'iter': [],
    'train_Accuracy': [], 'val_Accuracy': [],
    'train_Macro_F1': [], 'val_Macro_F1': [],
    'train_AUC': [], 'val_AUC': [],
    'train_QWK': [], 'val_QWK': [],
}

for it in range(1, best_iter + 1, curve_step):
    raw_tr = model.predict(X_train_flat, num_iteration=it, raw_score=True)
    raw_va = model.predict(X_val_flat, num_iteration=it, raw_score=True)

    proba_tr = softmax_2d(raw_tr)
    proba_va = softmax_2d(raw_va)
    pred_tr = np.argmax(proba_tr, axis=1)
    pred_va = np.argmax(proba_va, axis=1)

    mtr = compute_metrics(y_train, pred_tr, proba_tr, n_classes)
    mva = compute_metrics(y_val, pred_va, proba_va, n_classes)

    history['iter'].append(it)
    for metric_name in ['Accuracy', 'Macro_F1', 'AUC', 'QWK']:
        history[f'train_{metric_name}'].append(float(mtr[metric_name]))
        history[f'val_{metric_name}'].append(float(mva[metric_name]))

history_df = pd.DataFrame(history)
history_path = ARTIFACT_DIR / 'lightgbm_training_history.csv'
history_df.to_csv(history_path, index=False)

raw_val = model.predict(X_val_flat, num_iteration=best_iter, raw_score=True)
raw_test = model.predict(X_test_flat, num_iteration=best_iter, raw_score=True)

proba_val = softmax_2d(raw_val)
proba_test = softmax_2d(raw_test)
pred_val = np.argmax(proba_val, axis=1)
pred_test = np.argmax(proba_test, axis=1)

val_metrics = compute_metrics(y_val, pred_val, proba_val, n_classes)
test_metrics = compute_metrics(y_test, pred_test, proba_test, n_classes)

report = pd.DataFrame([
    {'Split': 'Val', **val_metrics},
    {'Split': 'Test', **test_metrics},
])
display(report)

out_path = ARTIFACT_DIR / 'lightgbm_ordinal_metrics.csv'
report.to_csv(out_path, index=False)
print('Saved:', out_path)
print('Saved:', history_path)

## Visualization: Training Curves

Biểu đồ training curves được chuẩn hóa theo format báo cáo khoa học với 4 metric chính: Accuracy, Macro F1, AUC, QWK.

In [ ]:
if 'history_df' not in globals():
    raise RuntimeError('Khong tim thay history_df. Hay chay lai cell huan luyen truoc.')

sns.set_theme(style='whitegrid', context='paper')
metrics = ['Accuracy', 'Macro_F1', 'AUC', 'QWK']
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=160, constrained_layout=True)
axes = axes.ravel()

for ax, metric in zip(axes, metrics):
    ax.plot(history_df['iter'], history_df[f'train_{metric}'], label='Train', linewidth=1.8, color='#1f77b4')
    ax.plot(history_df['iter'], history_df[f'val_{metric}'], label='Validation', linewidth=1.8, color='#d62728')
    ax.set_title(metric, fontsize=11, fontweight='semibold')
    ax.set_xlabel('Boosting Iteration')
    ax.set_ylabel(metric)
    ax.grid(True, linestyle='--', alpha=0.35)
    ax.legend(frameon=True, fontsize=9)

fig.suptitle('LightGBM Training Curves (Multiclass + Ordinal Penalty)', fontsize=13, fontweight='bold')
curve_path = ARTIFACT_DIR / 'lightgbm_training_curves.png'
fig.savefig(curve_path, dpi=300, bbox_inches='tight')
plt.show()

print('Saved:', curve_path)